### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex_client import QuinexClient
from quinex_application.quinex_postprocessing import QuinexPostprocessing

In [2]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2025_abstract")
qpost = QuinexPostprocessing()

In [ ]:
analysis = mongodb.get_collection("papers_2025_abstract")

In [ ]:
analysis[1]["analysis_result"]["annotations"]

In [ ]:
from bson import json_util
with open("papers_wos_abstract.json", "w", encoding="utf-8") as f:
    f.write(json_util.dumps(analysis, indent=2, ensure_ascii=False))

### **Load Analysis Metadata into DataFrame**

In [ ]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex_meta = qpost.create_metadata_df(analysis, meta_df)

In [ ]:
df_path = "../data/quinex/df_quinex_metadata.parquet"
qpost.save_df(df_quinex_meta, df_path)

### **Load Analysis Results into DataFrame**

In [7]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex = qpost.create_information_df(analysis, meta_df)

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
qpost.save_df(df_quinex, df_path)

### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with TechDB Hirarchy**

In [ ]:
df_quinex = qpost.load_df("../data/quinex/df_quinex.parquet")

In [ ]:
df_quinex_2015 = df_quinex[df_quinex["publication_year"] == 2015]

In [ ]:
df_quinex_2015 = df_quinex_2015.reset_index(drop=True)

In [ ]:
import requests
url = "http://127.0.0.1:8000/levenshtein/match_dataset"
json_dict = {
    "levenshtein": "similarity",
    "data": df_quinex_2015[['entity', 'property']].to_dict(orient="records")
}
response = requests.post(url, json=json_dict)

In [ ]:
import pandas as pd
df_response = pd.DataFrame(response.json())

columns_to_add = ['entity_mapping', 'entity_match_score', 'property_mapping', 'property_match_score']
df_quinex_2015[columns_to_add] = df_response[columns_to_add]

In [ ]:
qpost.save_df(df_quinex, "../data/quinex/df_quinex_levenshtein.parquet")

In [ ]:
df_levenshtein_2015 = df_quinex_2015[(df_quinex_2015['entity_match_score'] >= 0.75) & (df_quinex_2015['property_match_score'] >=0.75)]
df_levenshtein_2015 = df_levenshtein_2015.reset_index(drop=True)

In [ ]:
print(len(df_levenshtein_2015))

In [ ]:
df_levenshtein_2015.to_excel("../data/quinex/df_quinex_prop_ent_mapped_2015.xlsx")

In [ ]:
qpost.save_df(df_levenshtein, "../data/quinex/df_quinex_prop_ent_mapped.parquet")

In [ ]:
print(df_quinex["modifier"].value_counts())

In [ ]:
print(df_levenshtein["entity_mapping"].nunique())
print(df_levenshtein["property_mapping"].nunique())